In [2]:
import pandas as pd
df=pd.read_excel("Healthcare Analysis Dataset.xlsx", sheet_name='Data') 
print(df.head())
print(df.describe())
print(df.info())
print(df.shape)
print(df.isnull().sum())

  Patient ID  Age  Gender Blood Type Medical Condition Date of Admission  \
0    ID-2621   64  Female         A+      Hypertension        2020-03-03   
1    ID-2623   18  Female         A+      Hypertension        2022-09-12   
2    ID-2624   57  Female         A+      Hypertension        2022-02-26   
3    ID-2634   77  Female         A+      Hypertension        2020-12-11   
4    ID-2720   81  Female         A+      Hypertension        2019-09-19   

              Doctor                    Hospital Insurance Provider  \
0   Jessica Marshall  Houston Methodist Hospital           Medicare   
1       Sean Jimenez  Houston Methodist Hospital           Medicare   
2  Jennifer Crawford  Houston Methodist Hospital           Medicare   
3    Tiffany Ramirez  Houston Methodist Hospital           Medicare   
4      Peter Roberts  Houston Methodist Hospital           Medicare   

   Billing Amount  Room Number Admission Type Discharge Date Medication  \
0    43913.983738          353       Elec

In [4]:
negative_billing=df[df["Billing Amount"] < 0]
print(f'تعداد ردیف های با هزینه ی منفی: {len(negative_billing)}')
print(f'درصد از کل داده: {len(negative_billing)/ len(df)*100:.2f}%')
negative_billing[['Patient ID', "Hospital", 'Billing Amount', 'Admission Type']].head(10)
print(f'تعداد ردیف های کاملا تکراری: {df.duplicated().sum()}')
print('توزیع بر اساس بیمارستان:')
print(negative_billing["Hospital"].value_counts())
print('\nتوزیع بر اساس نوع پذیرش:')
print(negative_billing["Admission Type"].value_counts())
print('\nتوزیع بر اساس بیمه')
print(negative_billing['Insurance Provider'].value_counts())
print('\nبازه ی مقادیر منفی:')
print(negative_billing['Billing Amount'].describe())

تعداد ردیف های با هزینه ی منفی: 108
درصد از کل داده: 0.19%
تعداد ردیف های کاملا تکراری: 0
توزیع بر اساس بیمارستان:
Hospital
Houston Methodist Hospital        40
Johns Hopkins Hospital            20
UCLA Medical Center               17
Massachusetts General Hospital     7
Cedars-Sinai Medical Center        6
Northwestern Memorial Hospital     5
Cleveland Clinic                   5
UCSF Medical Center                4
Mayo Clinic                        3
NewYork-Presbyterian Hospital      1
Name: count, dtype: int64

توزیع بر اساس نوع پذیرش:
Admission Type
Urgent       38
Elective     37
Emergency    33
Name: count, dtype: int64

توزیع بر اساس بیمه
Insurance Provider
Medicare            56
UnitedHealthCare    27
Cigna               18
Aetna                7
Name: count, dtype: int64

بازه ی مقادیر منفی:
count     108.000000
mean     -499.301242
std       426.463202
min     -2008.492140
25%      -792.084802
50%      -369.093817
75%      -153.079565
max       -23.866729
Name: Billing Amoun

In [5]:
total_by_hospital=df['Hospital'].value_counts()
negative_by_hospital=negative_billing['Hospital'].value_counts()
proportion_hospital=(negative_by_hospital/ total_by_hospital *100).sort_values(ascending=False)
print('درصد منفی از کل هر بیمارستان:')
print(proportion_hospital.round(2))

total_by_insurance=df['Insurance Provider'].value_counts()
negative_by_insurance=negative_billing['Insurance Provider'].value_counts()
proportion_insurance=(negative_by_insurance/ total_by_insurance *100).sort_values(ascending=False)
print('\nدرصد منفی از کل هر بیمه:')
print(proportion_insurance.round(2))
print('آمار مقادیر مثبت Billing Amount:')
print(df[df['Billing Amount']> 0]['Billing Amount'].describe())

درصد منفی از کل هر بیمارستان:
Hospital
Massachusetts General Hospital    0.28
Cedars-Sinai Medical Center       0.25
UCLA Medical Center               0.25
Cleveland Clinic                  0.21
Northwestern Memorial Hospital    0.20
Houston Methodist Hospital        0.20
Johns Hopkins Hospital            0.18
UCSF Medical Center               0.16
Mayo Clinic                       0.12
NewYork-Presbyterian Hospital     0.04
Name: count, dtype: float64

درصد منفی از کل هر بیمه:
Insurance Provider
Cigna               0.32
Medicare            0.20
UnitedHealthCare    0.16
Aetna               0.13
Name: count, dtype: float64
آمار مقادیر مثبت Billing Amount:
count    55392.000000
mean     25590.084632
std      14178.657522
min          9.238787
25%      13297.478682
50%      25573.823359
75%      37849.210060
max      52764.276736
Name: Billing Amount, dtype: float64


In [6]:
df_billing=df[df['Billing Amount']> 0].copy()
df_adjustment=df[df['Billing Amount']< 0].copy()
print(f'تعداد رکورد برای تحلیل اصلی: {len(df_billing)}')
print(f'تعداد رکورد تعدیل/بازپرداخت(گزارش جدا): {len(df_adjustment)}')

تعداد رکورد برای تحلیل اصلی: 55392
تعداد رکورد تعدیل/بازپرداخت(گزارش جدا): 108


In [10]:
df_billing["Date of Admission"]=pd.to_datetime(df_billing["Date of Admission"])
df_billing['Discharge Date']=pd.to_datetime(df_billing['Discharge Date'])
df_billing['Length Of Stay']=(df_billing['Discharge Date']- df_billing['Date of Admission']).dt.days
print(df_billing['Length Of Stay'].describe())


negative_los=df_billing[df_billing["Length Of Stay"]<0]
print(f'تعداد ردیف با مدت بستری منفی: {len(negative_los)}')
zero_los=df_billing[df_billing["Length Of Stay"]==0]
print(f'تعداد ردیف با مدت بستری صفر: {len(zero_los)}')
print(f'درصد از کل: {len(zero_los)/len(df_billing)*100:.2f}%')

count    55392.000000
mean        15.508774
std          8.659552
min          1.000000
25%          8.000000
50%         15.000000
75%         23.000000
max         30.000000
Name: Length Of Stay, dtype: float64
تعداد ردیف با مدت بستری منفی: 0
تعداد ردیف با مدت بستری صفر: 0
درصد از کل: 0.00%


In [12]:
df_billing['Cost per Day']=df_billing['Billing Amount']/ df_billing['Length Of Stay']
print(df_billing['Cost per Day'].describe())
kpi_table=df_billing.groupby(['Hospital', 'Admission Type'])['Cost per Day'].mean().round(2)
print(kpi_table)
hospital_avg=df_billing.groupby('Hospital')['Cost per Day'].mean().sort_values(ascending=False)
overall_avg=df_billing['Cost per Day'].mean()
print(f'میانگین کلی Cost per Day:{overall_avg:.2f}')
print('\n میانگین هر بیمارستان (مرتب شده):')
print(hospital_avg.round(2))
print('\n انحراف از میانگین کلی:')
print((hospital_avg - overall_avg).round(2))

count    55392.000000
mean      3393.113514
std       5788.598892
min          0.949117
25%        858.766418
50%       1651.037393
75%       3206.552990
max      52211.852966
Name: Cost per Day, dtype: float64
Hospital                        Admission Type
Cedars-Sinai Medical Center     Elective          3440.14
                                Emergency         3396.28
                                Urgent            3517.23
Cleveland Clinic                Elective          3006.85
                                Emergency         3095.37
                                Urgent            3413.24
Houston Methodist Hospital      Elective          3555.31
                                Emergency         3299.14
                                Urgent            3503.73
Johns Hopkins Hospital          Elective          3294.07
                                Emergency         3385.82
                                Urgent            3390.56
Massachusetts General Hospital  Elective      

In [16]:
import numpy as np
hospital_stats=df_billing.groupby('Hospital')['Cost per Day'].agg(['mean', 'std', 'count'])
hospital_stats["SE"]=hospital_stats['std']/ np.sqrt(hospital_stats['count'])
overall_avg=df_billing['Cost per Day'].mean()
hospital_stats["z_score"]=(hospital_stats['mean']- overall_avg)/hospital_stats['SE']
print(hospital_stats.round(2).sort_values('z_score'))


short_stay_high_cost=df_billing[df_billing['Length Of Stay']<=2].sort_values('Cost per Day', ascending=False)
print(short_stay_high_cost[['Length Of Stay', 'Billing Amount', 'Cost per Day']].head(10))
comparison=df_billing.groupby('Hospital')['Cost per Day'].agg(['mean', 'median'])
print(comparison.round(2))

                                   mean      std  count      SE  z_score
Hospital                                                                
Cleveland Clinic                3172.28  5260.11   2419  106.95    -2.06
UCLA Medical Center             3317.41  5531.56   6836   66.90    -1.13
NewYork-Presbyterian Hospital   3299.27  5570.84   2333  115.34    -0.81
Johns Hopkins Hospital          3356.43  5695.21  11248   53.70    -0.68
Northwestern Memorial Hospital  3351.77  5766.98   2506  115.20    -0.36
Mayo Clinic                     3384.34  5777.54   2397  118.01    -0.07
Cedars-Sinai Medical Center     3450.66  5936.27   2399  121.20     0.47
UCSF Medical Center             3462.80  5977.25   2428  121.30     0.57
Massachusetts General Hospital  3499.34  5965.06   2464  120.17     0.88
Houston Methodist Hospital      3453.96  5948.05  20362   41.68     1.46
       Length Of Stay  Billing Amount  Cost per Day
19996               1    52211.852966  52211.852966
6716                

In [18]:
condition_stats=df_billing.groupby('Medical Condition')['Cost per Day'].agg(['mean', 'median', 'std', 'count'])
condition_stats["SE"]=condition_stats['std']/ np.sqrt(condition_stats['count'])
overall_avg=df_billing['Cost per Day'].mean()
condition_stats['z_score']=(condition_stats['mean']- overall_avg)/ condition_stats["SE"]
print(condition_stats.round(2).sort_values('z_score'))

                      mean   median      std  count     SE  z_score
Medical Condition                                                  
Arthritis          3270.72  1640.85  5556.34   5540  74.65    -1.64
Obesity            3365.64  1668.86  5663.67  12739  50.18    -0.55
Hypertension       3398.52  1636.73  5850.35  13852  49.71     0.11
Cancer             3411.82  1643.56  5810.31   5543  78.04     0.24
Diabetes           3426.30  1650.67  5897.57  13846  50.12     0.66
Asthma             3493.84  1660.02  5874.00   3872  94.40     1.07


In [19]:
import numpy as np
def test_significance(df, group_col, value_col='Cost per Day'):
    stats=df.groupby(group_col)[value_col].agg(['mean', 'std', 'count'])
    stats['SE']=stats['std']/ np.sqrt(stats['count'])
    overall_avg=df[value_col].mean()
    stats['z_score']=(stats['mean']- overall_avg)/ stats['SE']
    return stats.round(2).sort_values('z_score')

for col in ['Admission Type', 'Insurance Provider', 'Gender', 'Blood Type', 'Test Results']:
    print(f'\n{'='*20} {col} {'='*20}')
    print(test_significance(df_billing, col))

print('\n همسبتگی Age با Cost per Day:', df_billing['Age'].corr(df_billing['Cost per Day']).round(3))
print('\n همسبتگی Age با Billing Amount:', df_billing['Age'].corr(df_billing['Billing Amount']).round(3))


==================== Admission Type ====================
                   mean      std  count     SE  z_score
Admission Type                                         
Emergency       3319.58  5649.56  18236  41.84    -1.76
Elective        3400.89  5821.93  18618  42.67     0.18
Urgent          3457.64  5888.78  18538  43.25     1.49

==================== Insurance Provider ====================
                       mean      std  count     SE  z_score
Insurance Provider                                         
Medicare            3351.65  5691.56  27694  34.20    -1.21
Cigna               3366.78  5582.63   5532  75.06    -0.35
Aetna               3376.04  5852.14   5543  78.60    -0.22
UnitedHealthCare    3476.64  5991.03  16623  46.47     1.80

==================== Gender ====================
               mean      std  count     SE  z_score
Gender                                             
Female      3351.65  5691.56  27694  34.20    -1.21
Non-binary  3366.78  5582.63   553

In [22]:
print('توزیع Admission Type:')
print(df_billing['Admission Type'].value_counts(normalize=True).round(3)*100)
print('\n درصد نتایج آزمایش غیر عادی به تفکیک بیمارستان:')
abnormal_rate=df_billing.groupby('Hospital')['Test Results'].apply(lambda x: (x=='Abnormal').mean()* 100)
print(abnormal_rate.round(1).sort_values(ascending=False))

توزیع Admission Type:
Admission Type
Elective     33.6
Urgent       33.5
Emergency    32.9
Name: proportion, dtype: float64

 درصد نتایج آزمایش غیر عادی به تفکیک بیمارستان:
Hospital
Cedars-Sinai Medical Center       55.8
Cleveland Clinic                  55.5
Mayo Clinic                       55.4
Massachusetts General Hospital    55.3
NewYork-Presbyterian Hospital     55.3
Houston Methodist Hospital        55.1
Johns Hopkins Hospital            54.9
UCLA Medical Center               54.7
Northwestern Memorial Hospital    54.2
UCSF Medical Center               54.1
Name: Test Results, dtype: float64


In [23]:
hospital_n=df_billing.groupby('Hospital').size()
hospital_abnormal_rate=df_billing.groupby('Hospital')['Test Results'].apply(lambda x: (x=='Abnormal').mean())
overall_rate=(df_billing['Test Results']== 'Abnormal').mean()
se_proportion=np.sqrt(overall_rate * (1- overall_rate) / hospital_n)
z_scores_proportion=(hospital_abnormal_rate- overall_rate)/ se_proportion

result=pd.DataFrame({
    'rate':(hospital_abnormal_rate *100).round(1),
    'z_score':z_scores_proportion.round(2)
}).sort_values('z_score')
print(result)

                                rate  z_score
Hospital                                     
UCSF Medical Center             54.1    -0.91
Northwestern Memorial Hospital  54.2    -0.81
UCLA Medical Center             54.7    -0.44
Johns Hopkins Hospital          54.9    -0.24
Houston Methodist Hospital      55.1     0.26
Massachusetts General Hospital  55.3     0.28
NewYork-Presbyterian Hospital   55.3     0.29
Mayo Clinic                     55.4     0.36
Cleveland Clinic                55.5     0.48
Cedars-Sinai Medical Center     55.8     0.81


In [25]:

df_billing.to_excel("healthcare_cleaned.xlsx", sheet_name="Cleaned Data", index=False)


with pd.ExcelWriter("healthcare_summary.xlsx") as writer:
    hospital_stats.to_excel(writer, sheet_name="Hospital Stats")
    condition_stats.to_excel(writer, sheet_name="Condition Stats")
    result.to_excel(writer, sheet_name="Abnormal Rate by Hospital")
    df_adjustment.to_excel(writer, sheet_name="Adjustments", index=False)